# ДЗ 1.1: HDFS + MapReduce

## Задание 1

In [1]:
%%writefile mapper.py
import sys

for line in sys.stdin:
    for word in line.split():
        if len(word) > 4:
            print(f"{word}\t1")

Writing mapper.py


In [2]:
%%writefile reducer.py
import sys

current_word = None
current_count = 0

for line in sys.stdin:
    word, count = line.strip().split('\t')
    if word == current_word:
        current_count += int(count)
    else:
        if current_word is not None:
            print(f"{current_word}\t{current_count}")
        current_word = word
        current_count = int(count)

if current_word is not None:
    print(f"{current_word}\t{current_count}")

Writing reducer.py


Локальная проверка:

In [3]:
! cat bible.txt | python3 mapper.py | sort | python3 reducer.py | sort -k2,2nr | head

shall	9658
which	4119
their	3801
there	1799
before	1701
against	1587
shalt	1586
children	1559
said,	1556
them,	1549
sort: write failed: 'standard output': Broken pipe
sort: write error


Запуск на Hadoop:

In [4]:
! hdfs dfs -mkdir -p /hw1/input
! hdfs dfs -put -f bible.txt Poseschenia_sai_774_tov.csv /hw1/input/
! hdfs dfs -ls /hw1/input
! hdfs dfs -rm -r -f /hw1/wordcount

Found 2 items
-rw-r--r--   1 hadoop users   36443383 2026-10-01 16:33 /hw1/input/Poseschenia_sai_774_tov.csv
-rw-r--r--   1 hadoop users    4047392 2026-10-01 16:33 /hw1/input/bible.txt


In [5]:
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="wordcount" \
-files mapper.py,reducer.py \
-mapper "python3 mapper.py" \
-reducer "python3 reducer.py" \
-input /hw1/input/bible.txt \
-output /hw1/wordcount

packageJobJar: [/tmp/hadoop-unjar8174309548089378735/] [] /tmp/streamjob13137431206973171902.jar tmpDir=null
2026-10-01 16:33:18,017 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.5:8032
2026-10-01 16:33:18,598 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.5:8032
2026-10-01 16:33:19,439 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790872319996_0001
2026-10-01 16:33:20,529 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-10-01 16:33:20,754 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-10-01 

In [6]:
! hdfs dfs -cat /hw1/wordcount/part-* | sort -k2,2nr | head

shall	9658
which	4119
their	3801
there	1799
before	1701
against	1587
shalt	1586
children	1559
said,	1556
them,	1549
sort: write failed: 'standard output': Broken pipe
sort: write error


## Задание 2

Две джобы:
1. считаем посещения для каждой пары (дата, сайт);
2. Hadoop сортирует по дате и по числу посещений (по убыванию), а reducer берёт первые 5 строк каждой даты.

Так reducer'ы хранят только текущий ключ и счётчик, память O(1).

In [7]:
%%writefile visits_mapper.py
import sys
import logging

logging.basicConfig(level=logging.INFO)

for line in sys.stdin:
    try:
        site, dt = line.strip().split(';')
        date = dt.split()[0]
        print(f"{date}\t{site}\t1")
    except Exception as e:
        logging.error(f"bad line {line!r}: {e}")

Writing visits_mapper.py


In [8]:
%%writefile visits_reducer.py
import sys

current_key = None
current_count = 0

for line in sys.stdin:
    date, site, count = line.strip().split('\t')
    if (date, site) == current_key:
        current_count += int(count)
    else:
        if current_key is not None:
            print(f"{current_key[0]}\t{current_key[1]}\t{current_count}")
        current_key = (date, site)
        current_count = int(count)

if current_key is not None:
    print(f"{current_key[0]}\t{current_key[1]}\t{current_count}")

Writing visits_reducer.py


In [9]:
%%writefile top5_reducer.py
import sys

current_date = None
n = 0

for line in sys.stdin:
    date, site, count = line.strip().split('\t')
    if date != current_date:
        current_date = date
        n = 0
    if n < 5:
        print(f"{date}\t{site}\t{count}")
        n += 1

Writing top5_reducer.py


In [10]:
! hdfs dfs -rm -r -f /hw1/visits /hw1/top5

In [11]:
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="visits_count" \
-D stream.num.map.output.key.fields=2 \
-files visits_mapper.py,visits_reducer.py \
-mapper "python3 visits_mapper.py" \
-reducer "python3 visits_reducer.py" \
-input /hw1/input/Poseschenia_sai_774_tov.csv \
-output /hw1/visits

packageJobJar: [/tmp/hadoop-unjar1861673539448431415/] [] /tmp/streamjob229092678070004023.jar tmpDir=null
2026-10-01 16:34:10,313 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.5:8032
2026-10-01 16:34:10,882 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.5:8032
2026-10-01 16:34:11,533 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790872319996_0002
2026-10-01 16:34:12,748 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-10-01 16:34:12,956 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-10-01 16

In [12]:
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="top5" \
-D mapreduce.job.reduces=1 \
-D stream.num.map.output.key.fields=3 \
-D mapreduce.job.output.key.comparator.class=org.apache.hadoop.mapreduce.lib.partition.KeyFieldBasedComparator \
-D mapreduce.partition.keycomparator.options="-k1,1 -k3,3nr" \
-files top5_reducer.py \
-mapper cat \
-reducer "python3 top5_reducer.py" \
-input /hw1/visits \
-output /hw1/top5

packageJobJar: [/tmp/hadoop-unjar17846369460051424160/] [] /tmp/streamjob8699393700540008668.jar tmpDir=null
2026-10-01 16:34:53,400 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.5:8032
2026-10-01 16:34:53,979 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.5:8032
2026-10-01 16:34:54,721 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790872319996_0003
2026-10-01 16:34:55,826 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-10-01 16:34:55,994 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-10-01 

In [13]:
! hdfs dfs -cat /hw1/top5/part-*

2024-05-26	https://gonzales-bautista.com/	335
2024-05-26	http://smith.com/	235
2024-05-26	https://www.smith.com/	221
2024-05-26	https://smith.com/	212
2024-05-26	http://www.smith.com/	212
2024-05-27	https://gonzales-bautista.com/	376
2024-05-27	https://www.smith.com/	270
2024-05-27	https://smith.com/	236
2024-05-27	http://smith.com/	215
2024-05-27	http://www.smith.com/	208
2024-05-28	https://gonzales-bautista.com/	368
2024-05-28	https://smith.com/	256
2024-05-28	https://www.smith.com/	251
2024-05-28	http://smith.com/	224
2024-05-28	http://www.smith.com/	204
2024-05-29	https://gonzales-bautista.com/	402
2024-05-29	https://www.smith.com/	242
2024-05-29	http://www.smith.com/	223
2024-05-29	https://smith.com/	220
2024-05-29	http://smith.com/	206
2024-05-30	https://gonzales-bautista.com/	353
2024-05-30	https://smith.com/	246
2024-05-30	https://www.smith.com/	239
2024-05-30	http://smith.com/	229
2024-05-30	http://www.smith.com/	225
2024-05-31	https://gonzales-bautista.com/	374
2024-05-31	htt